# Benchmark VMTEB-ALQAC (Model LOCAL) cho AI_RAG_LEGAL trên Kaggle

Metric chính: **Macro-F2 + MRR** (620 query, 13 batch x 50).

Dùng **model local** (Qwen instruct, 4-bit/8-bit chay tren GPU Kaggle T4/P100 16GB) — KHONG can Google API key.

## Setup trước khi chạy (một lần, ngoài Kaggle):
1. `python scripts/package_kaggle_data.py --mode both` -> tạo kaggle_assets/data_pkg/ + data.zip
2. Upload lên Kaggle Dataset tên `ai-rag-legal-data`
3. Push code (src/, scripts/, requirements-kaggle.txt) lên GitHub

## Trong notebook:
- Add Input -> dataset **ai-rag-legal-data**
- Chọn Accelerator GPU (T4 x2 / P100)
- Chọn model local phù hợp VRAM: Qwen3-8B (fp16 ~16GB vừa T4) hoặc Qwen3.5-4B (4bit ~4GB)

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "AI_RAG_LEGAL"

# 1. Clone repo (thay URL github cua ban)
GIT_URL = os.environ.get("GIT_REPO_URL", "https://github.com/NosKaiser/AI_RAG_LEGAL.git")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)

# 2. sys.path trick: bien Kaggle thanh moi truong local
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["HF_HOME"] = str(WORK / "hf_cache")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
print("Repo:", REPO)
print("Python:", sys.executable)

In [ ]:
# 3. Giai nen data tu Kaggle Input vao data/
DATA_DIR = REPO / "data"
DATA_DIR.mkdir(exist_ok=True)

input_root = Path("/kaggle/input")
candidates = [p for p in input_root.glob("ai-rag-legal-data*")]
if not candidates:
    raise RuntimeError("Chua attach dataset ai-rag-legal-data! Add Input -> ai-rag-legal-data")
src_pkg = candidates[0]
print("Dataset path:", src_pkg)

zips = list(src_pkg.glob("*.zip"))
if zips:
    for z in zips:
        print(f"Unzip {z.name}...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_DIR)
else:
    for sub in ("indexes", "processed"):
        s = src_pkg / sub
        d = DATA_DIR / sub
        if s.exists():
            shutil.copytree(s, d, dirs_exist_ok=True)

idx = DATA_DIR / "indexes" / "dense.index"
corpus = DATA_DIR / "processed" / "corpus.jsonl"
print("dense.index:", idx.exists())
print("corpus.jsonl:", corpus.exists())

In [ ]:
# 4. Tao .env cho local: device cuda, model embedding/reranker open-source
(REPO / ".env").write_text("\n".join([
    "EMBEDDING_MODEL=mainguyen9/vietlegal-harrier-0.6b",
    "EMBEDDING_DIM=1024",
    "RERANKER_MODEL=AITeamVN/Vietnamese_Reranker",
    "DEVICE=cuda",
    "RETRIEVAL_TOP_K=500",
    "RERANK_TOP_K=50",
    "FINAL_TOP_K=20",
    "SCORE_THRESHOLD=0.7",
    "MAX_TOKENS=2048",
    "TEMPERATURE=0.1",
    "DATA_DIR=data",
    "INDEX_DIR=data/indexes",
]), encoding="utf-8")
print(".env created")

# Local model mac dinh": "Qwen/Qwen3.5-4B-Instruct",
# P100/T4: 4B tren 4-bit chay OK. Neu muon hon nua chon Qwen3-8B fp16/x4bit
LLM_MODEL = "Qwen/Qwen3.5-4B-Instruct"

In [ ]:
# 5. Cai dependencies (torch/cuda co san tren Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "transformers", "accelerate", "datasets", "faiss-cpu", "bm25s",
    "scikit-learn", "scipy", "tqdm", "pydantic-settings", "bitsandbytes",
], check=True)
print("Dependencies OK")

## 6. Chạy benchmark — model LOCAL

**Test nhanh 3 query trước** (không có script riêng; dùng batch idx 0, batch-size 5):

```bash
python -m scripts.evaluate_vmteb_batch --batch-idx 0 --batch-size 5 --workers 1 --llm hf --hf-model Qwen/Qwen3.5-4B-Instruct
```

Lưu ý model local: workers nên = 1 (1 GPU, suy luận tuần tự). Batch đầy đủ 50 query/ batch:

```bash
python -m scripts.evaluate_vmteb_batch --batch-idx 0 --batch-size 50 --workers 1 --llm hf
```

**Toàn bộ 13 batch** (620 query, model local chậm hơn Gemini — ước tính 6-10/s

```bash
python scripts/run_all_batches.py --start 0 --end 12 --llm hf --hf-model Qwen/Qwen3.5-4B-Instruct
```

**Resume batch lỗi**:

```bash
python -m scripts.evaluate_vmteb_resume --batch-idx N --batch-size 50 --workers 1 --llm hf
```

Kết quả: `data/results/vmteb_batch{N}_metrics.json` (+ _detail.json, _trace.json)

In [ ]:
# Test 5 query dau.
ret = subprocess.run([sys.executable, "-m", "scripts.evaluate_vmteb_batch",
    "--batch-idx", "0", "--batch-size", "5", "--workers", "1",
    "--llm", "hf", "--hf-model", LLM_MODEL], cwd=REPO)
print("Exit:", ret.returncode)

In [ ]:
# Full benchmark (620 query). Model local nen co the chia nhieu session:
# lan 1: --start 0 --end 6; lan 2: --start 7 --end 12
ret = subprocess.run([sys.executable, "scripts/run_all_batches.py", "--start", "0", "--end", "12",
    "--llm", "hf", "--hf-model", LLM_MODEL], cwd=REPO)
print("Exit:", ret.returncode)

In [ ]:
# Xem tong hop diem
import json
res_dir = DATA_DIR / "results"
rows = []
for f in sorted(res_dir.glob("vmteb_batch*_metrics.json")):
    m = json.loads(f.read_text(encoding="utf-8"))
    rows.append((f.stem, m.get("num_queries"), m.get("num_errors"), m.get("macro_f2"), m.get("micro_f2"), m.get("mrr"), m.get("recall@5")))
import pandas as pd
df = pd.DataFrame(rows, columns=["batch", "n", "err", "macro_f2", "micro_f2", "mrr", "recall@5"])
display(df)
print(df[['macro_f2','micro_f2','mrr','recall@5']].mean())
df.to_csv("data/results/vmteb_summary.csv", index=False)
print("Saved: data/results/vmteb_summary.csv")